### Notebook collecting commands/scripts which are run after fits/scans complete

In [ ]:
# imports
# autoreload magic:
%load_ext autoreload
%autoreload 2
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.stats import norm
%matplotlib inline
import sys

from NNMFit.utilities import load_pickle
from NNMFit.utilities import ScanHandler

unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
sys.path.append(unblind_script_path)
from freefit_parameter_config import FreefitParamConfig

run scan analysis: calculate MC Histogram, Data Histogram, and Saturated LLH of the best fit

In [ ]:
fit_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/data_fits"

saturated_path   = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind/saturated"

config_path      = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
config_input_path = f"{config_path}/analysis_configs/data/hese/step2_hese_flavor_reunblind/input_pars_only"
config_plot_path  = f"{config_path}/analysis_configs/data/hese/step2_hese_flavor_reunblind/plotting"
config_pse_path   = f"{config_path}/analysis_configs/pse/hese/step2_hese_flavor_reunblind"

os.system(f"mkdir -p {saturated_path}")
os.system(f"mkdir -p {config_input_path}")
os.system(f"mkdir -p {config_plot_path}")
os.system(f"mkdir -p {config_pse_path}")

In [ ]:
scans = {
    # fit with the PassingHESE passing fractions, see dag_scans/.../step2_hese_flavor_reunblind/data_fits.sh
    # without detector priors (`priors` key was not read by NNMFit):
    # "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE": f"{fit_path}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE",
    # with the detector priors applied (fixed `prior` key)
    "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_fixedprior":
        f"{fit_path}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_fixedprior",
}

In [ ]:
for name, scan_dir in scans.items():
    # run do_scan_analysis.py with scan_dir as arg:
    print(f"Processing scan directory: {scan_dir}")
    os.system(f"python {unblind_script_path}/do_scan_analysis.py {scan_dir} --sat_llh_save_dir {saturated_path}/{name}")

Collect the bestfit parameter values in a config file (for later pseudoexperiments etc.)

In [ ]:
# find the best-fit for each scan and write a configuration file using the
# bestfit params as input params:
freefit_config_names = {}
min_llh_dict = {}
scan_llh = {}
for name, scan_path in scans.items():
    print(scan_path)
    freefit_config_hdl = FreefitParamConfig(scan_path=scan_path)
    print(freefit_config_hdl.get_name())
    freefit_config_hdl.write(custom_name=f"{config_input_path}/{name}.yaml")
    llh = freefit_config_hdl.get_min_llh_freefit(freefit_config_hdl.scan_hdl)
    min_llh_dict[freefit_config_hdl.get_name()] = llh
    scan_llh[name] = llh
    freefit_config_names[os.path.basename(os.path.dirname(scan_path))
                        ] = freefit_config_hdl.get_name()


In [ ]:
# For each scan, create a subfolder with full data configs (separate + combined detectors)
# and component-only variants (Astro / per-flavor Astro / Conv / Prompt / Muon / MuonConv / MuonPrompt).
import yaml
import copy
import json

DETECTOR_CONFIGS_SEPARATE = [
    "IC86_pass2_SnowStorm_FTP_HESE_Cascades",
    "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades",
    "IC86_pass2_SnowStorm_FTP_HESE_Tracks",
]
DETECTOR_CONFIGS_COMBINED = ["IC86_pass2_SnowStorm_FTP_HESE_Combined"]


def flavor_ratios_to_angles(fr0, fr1, fr2):
    norm = fr0 + fr1 + fr2
    fr0_n, fr1_n, fr2_n = fr0 / norm, fr1 / norm, fr2 / norm
    sphi4 = (1.0 - fr2_n) ** 2
    sphi2 = 1.0 - fr2_n
    if sphi2 == 0.0:
        c2psi = 0.0
    else:
        x = np.sqrt(fr0_n / sphi2)
        if x > 1.0:
            x = 1.0
        c2psi = np.cos(2.0 * np.arccos(x))
    return sphi4, c2psi   # a = sphi4,  b = c2psi


def write_data_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: data\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


def write_pse_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: pseudoexp\n")
        f.write("pseudoexp_type: poisson\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


for scan_name, scan_path in scans.items():
    input_yaml_path = f"{config_input_path}/{scan_name}.yaml"
    with open(input_yaml_path) as f:
        base_data = yaml.safe_load(f)
    params = base_data["input_params"]

    astro_norm        = params["astro_norm"]
    astro_nue_ratio   = params["astro_nue_ratio"]
    astro_nutau_ratio = params["astro_nutau_ratio"]

    # Convert fitted ratios to flavor fractions, then to (a, b) angle parameters
    f_e   = astro_nue_ratio   / (1 + astro_nue_ratio + astro_nutau_ratio)
    f_mu  = 1.0               / (1 + astro_nue_ratio + astro_nutau_ratio)
    f_tau = astro_nutau_ratio / (1 + astro_nue_ratio + astro_nutau_ratio)
    a, b  = flavor_ratios_to_angles(f_e, f_mu, f_tau)

    # Total normalization = NuMu (×1) + NuE (×nue_ratio) + NuTau (×nutau_ratio)
    total_astro_norm = astro_norm * (1 + astro_nue_ratio + astro_nutau_ratio)

    params["total_astro_norm"] = total_astro_norm
    params["gamma"] = params["gamma_astro"]
    params["a"] = a
    params["b"] = b
    print(f"[{scan_name}] astro_nue_ratio={astro_nue_ratio:.4f}  astro_nutau_ratio={astro_nutau_ratio:.4f}")
    print(f"             f_e={f_e:.4f}  f_mu={f_mu:.4f}  f_tau={f_tau:.4f}")
    print(f"             a={a:.6f}  b={b:.6f}  total_astro_norm={total_astro_norm:.6f}")

    # all flux normalizations, every component switches off all but its own
    ALL_NORMS_OFF = {"astro_norm": 0, "total_astro_norm": 0, "conv_norm": 0, "prompt_norm": 0, "combined_muongun_norm": 0}

    # split the combined muon template into its conv/prompt parts:
    # mu = N*[(1+a)*T_conv + (1-a)*T_prompt], so the conv part is reproduced by
    # norm N(1+a)/2 at a=+1, and the prompt part by norm N(1-a)/2 at a=-1
    muon_norm = params["combined_muongun_norm"]
    muon_asym = params["muongun_asymmetry"]

    # Astro        — astro_SPL_no_inel.yaml     (ratio params, best-fit composition)
    # Astro_3flavor — astro_SPL_3flavor_no_inel.yaml (angle params, best-fit composition)
    # Flavor components — pure-flavor a/b angles, total_astro_norm scaled by each flavor's ratio:
    #   NuE: (a=1, b=1),  NuMu: (a=1, b=-1),  NuTau: (a=0, b=0)
    #   total_astro_norm: astro_norm × nue_ratio / × 1 / × nutau_ratio
    COMPONENT_OVERRIDES = {
        "Astro":         {**ALL_NORMS_OFF, "astro_norm": astro_norm},
        "Astro_3flavor": {**ALL_NORMS_OFF, "total_astro_norm": total_astro_norm,                "a": a,   "b": b},
        "Astro_NuE":     {**ALL_NORMS_OFF, "total_astro_norm": astro_norm * astro_nue_ratio,   "a": 1.0, "b": 1.0},
        "Astro_NuMu":    {**ALL_NORMS_OFF, "total_astro_norm": astro_norm * 1.0,               "a": 1.0, "b": -1.0},
        "Astro_NuTau":   {**ALL_NORMS_OFF, "total_astro_norm": astro_norm * astro_nutau_ratio, "a": 0.0, "b": 0.0},
        "Conv":          {**ALL_NORMS_OFF, "conv_norm": params["conv_norm"]},
        "Prompt":        {**ALL_NORMS_OFF, "prompt_norm": params["prompt_norm"]},
        "Muon":          {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm},
        "MuonConv":      {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm * (1 + muon_asym) / 2, "muongun_asymmetry": 1.0},
        "MuonPrompt":    {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm * (1 - muon_asym) / 2, "muongun_asymmetry": -1.0},
    }

    # pse configs
    write_pse_config(params, DETECTOR_CONFIGS_SEPARATE, f"{config_pse_path}/{scan_name}.yaml")
    print(f"Written: {config_pse_path}/{scan_name}.yaml")

    # data configs for plotting
    scan_dir = f"{config_plot_path}/{scan_name}"
    os.makedirs(scan_dir, exist_ok=True)

    variants = [
        (scan_name,               DETECTOR_CONFIGS_SEPARATE),
        (f"{scan_name}_combined", DETECTOR_CONFIGS_COMBINED),
    ]

    for stem, det_configs in variants:
        write_data_config(params, det_configs, f"{scan_dir}/{stem}.yaml")
        print(f"Written: {scan_dir}/{stem}.yaml")

        for component, overrides in COMPONENT_OVERRIDES.items():
            write_data_config(params, det_configs, f"{scan_dir}/{stem}_{component}.yaml", overrides=overrides)
            print(f"Written: {scan_dir}/{stem}_{component}.yaml")

    # best fit with one detector systematic at a time scanned over its fit range (5 steps + its best fit),
    # all other detector systematics at nominal (prior centre), separate detectors only
    with open(f"{scan_path}/Fit_Configuration.yaml") as f:
        fit_config = yaml.safe_load(f)
    snowstorm = next(b for b in fit_config["config"].values() if isinstance(b, dict) and b.get("class") == "SnowStormGradient")
    det_names = [p.strip() for p in snowstorm["parameters"].split(",")]
    det_nominal = {p: float(v) for p, v in zip(det_names, str(snowstorm["default"]).split(","))}
    det_ranges = {p: tuple(float(x) for x in r.strip().strip("()").split(",")) for p, r in zip(det_names, snowstorm["range"].split("|"))}

    det_values = {}
    for p in det_names:
        lo, hi = det_ranges[p]
        det_values[p] = sorted({float(f"{v:.4g}") for v in [*np.linspace(lo, hi, 5), params[p]]})

    for p, values in det_values.items():
        for v in values:
            det_overrides = {**det_nominal, p: v}
            stem = f"{scan_name}_detsyst_{p}_{v:.4g}"
            write_data_config(params, DETECTOR_CONFIGS_SEPARATE, f"{scan_dir}/{stem}.yaml", overrides=det_overrides)
            for component, overrides in COMPONENT_OVERRIDES.items():
                write_data_config(params, DETECTOR_CONFIGS_SEPARATE, f"{scan_dir}/{stem}_{component}.yaml",
                                  overrides={**overrides, **det_overrides})
        print(f"Written: {scan_dir}/{scan_name}_detsyst_{p}_*.yaml for {p} = {values} (best fit {params[p]:.4g})")

    # used by reco_space_graphs.ipynb and plot_configs.py
    with open(f"{scan_dir}/{scan_name}_detsyst_values.json", "w") as f:
        json.dump({"values": det_values, "bestfit": {p: params[p] for p in det_names}, "nominal": det_nominal}, f, indent=1)
    print(f"Written: {scan_dir}/{scan_name}_detsyst_values.json")
